# Embedding approaches

Dense retrieval on the same chunks and the same questions. Only the text that is embedded changes.

| Approach | What is embedded | What is stored |
| --- | --- | --- |
| Original | Chunk text | Chunk text |
| Summary | Seeded short summary | Chunk text and that summary |
| Contextual | Summary, then the chunk text | Chunk text and that summary. This is the default. |

Seeded summaries are the Summary section already written for each document. Qwen is not called for this plot. Incident status still comes from SQLite.

The first run downloads `BAAI/bge-small-en-v1.5`.

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import security_investigator.evaluation as evaluation

importlib.reload(evaluation)
from security_investigator.config import Settings
from security_investigator.evaluation import generate_benchmark, score_embedding_approaches
from security_investigator.generator import generate_company
from security_investigator.ingestion import (
    build_chunks,
    knowledge_documents,
    load_operational_documents,
    sentence_transformer_embedder,
)

settings = Settings(_env_file=ROOT / ".env")
company = generate_company(seed=settings.random_seed, reference_time=settings.reference_time)
questions = generate_benchmark(company.manifest)
database = ROOT / settings.database_path
if not database.exists():
    raise SystemExit("database is missing; run scripts/seed_database.py first")
chunks = build_chunks(
    knowledge_documents(settings) + load_operational_documents(database, settings)
)
print(f"{len(chunks)} chunks, {len(questions)} questions")

In [ ]:
ready = sum(1 for chunk in chunks if chunk.get("summary"))
print(f"seeded summaries: {ready}/{len(chunks)}")

In [ ]:
embedder = sentence_transformer_embedder(settings.embedding_model)
scores = await score_embedding_approaches(chunks, questions, embedder, settings=settings)
labels = {
    "original": "Original chunk",
    "summary": "Summary only",
    "contextual": "Summary + original",
}
for row in scores:
    print(
        f"{labels[row['approach']]:22}  "
        f"recall {row['recall_at_5']:.3f}  "
        f"precision {row['precision_at_5']:.3f}  "
        f"mrr {row['mrr']:.3f}  "
        f"ndcg {row['ndcg_at_5']:.3f}"
    )

In [ ]:
metrics = ["recall_at_5", "precision_at_5", "mrr", "ndcg_at_5"]
metric_labels = ["Recall@5", "Precision@5", "MRR", "nDCG@5"]
colors = ["#4C78A8", "#F58518", "#54A24B"]
fig, ax = plt.subplots(figsize=(9, 4.8))
positions = range(len(metrics))
width = 0.24
for index, row in enumerate(scores):
    offset = (index - (len(scores) - 1) / 2) * width
    bars = ax.bar(
        [pos + offset for pos in positions],
        [row[metric] for metric in metrics],
        width=width,
        label=labels[row["approach"]],
        color=colors[index],
    )
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
ax.set_xticks(list(positions), metric_labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title(f"Dense retrieval on {len(questions)} questions")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()